In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# BigQuery DataFrames - Python UDF Transpiler (Experimental)

This notebook introduces the experimental **Python-to-SQL UDF Transpiler** in BigQuery DataFrames (`bigframes`).

## Overview

BigQuery DataFrames allows you to define custom scalar logic in Python and apply it to your dataframes (e.g. using `df.apply()` or `series.map()`). By default, doing so deploys your code as a Google Cloud Function and integrates it via BigQuery Remote Functions. While powerful, this introduces deployment latency and configuration overhead.

The **Python UDF Transpiler** solves this. If enabled, it analyzes the Python bytecode of your custom function and **transpiles** it directly into equivalent native BigQuery SQL expressions at runtime.

This means you can write standard Python/lambda functions and execute them in BigQuery with:
1. **Zero deployment overhead** (no Cloud Functions created).
2. **Native query performance** (runs directly inside the BigQuery execution engine).

This notebook showcases the transpiler across various use cases, including element-wise transforms, custom aggregations (`groupby.agg`), and window/broadcasting operations (`groupby.transform`).

---

### Key Features demonstrated:
1. **Element-wise mapping** using standard python functions (with nested conditionals and string f-strings).
2. **Groupby Aggregation** using custom Python aggregate logic.
3. **Groupby Transform** using custom Python windowing/broadcasting logic.


## Setup & Enable Transpiler

To use the Python UDF transpiler, we enable the experimental option:
`bpd.options.experiments.enable_python_transpiler = True`


In [ ]:
import bigframes.pandas as bpd

# Configure the BigQuery project to 'bigframes-dev' for this demo
bpd.options.bigquery.project = "bigframes-dev"
bpd.options.bigquery.ordering_mode = 'partial'

# Enable the Python UDF Transpiler!
bpd.options.experiments.enable_python_transpiler = True

session = bpd.get_global_session()
print(f"Using Google Cloud Project: {session.bqclient.project}")
print(f"Python transpiler enabled: {bpd.options.experiments.enable_python_transpiler}")


/usr/local/google/home/tbergeron/src/google-cloud-python/packages/bigframes/bigframes/_config/experiment_options.py:182: PythonTranspilerPreviewWarning: Python transpiler is an unstable, experimental feature, and not yet
fully validated, use at your own risk.
  warnings.warn(msg, category=bfe.PythonTranspilerPreviewWarning)
/usr/local/google/home/tbergeron/src/google-cloud-python/packages/bigframes/bigframes/core/global_session.py:113: DefaultLocationWarning: No explicit location is set, so using location US for the session.
  _global_session = bigframes.session.connect(


Using Google Cloud Project: bigframes-dev
Python transpiler enabled: True


## 1. Element-wise Mapping (UDF Transpilation)

Let's start with a basic example: applying a custom Python function to columns/series element-wise.

We'll load the public penguins dataset and use a Python function containing arithmetic operations, conditional logic, and f-string formatting.


In [3]:
# Load the public penguins table
df = bpd.read_gbq("bigquery-public-data.ml_datasets.penguins")

# Define a Python function containing conditionals and f-strings
def classify_weight(weight):
    if weight is None:
        return "Unknown"
    elif weight > 4500:
        return f"Heavy ({weight}g)"
    else:
        return f"Light ({weight}g)"

# Apply the Python function (this will be transpiled directly to a SQL CASE expression!)
df_classified = df.assign(
    weight_class=df["body_mass_g"].apply(classify_weight)
)

# Display the results
df_classified[["species", "body_mass_g", "weight_class"]].head(10)


,species,body_mass_g,weight_class
0,Gentoo penguin (Pygoscelis papua),5950.0,Heavy (5950g)
1,Gentoo penguin (Pygoscelis papua),4400.0,Light (4400g)
2,Adelie Penguin (Pygoscelis adeliae),3075.0,Light (3075g)
3,Gentoo penguin (Pygoscelis papua),5000.0,Heavy (5000g)
4,Gentoo penguin (Pygoscelis papua),4500.0,Light (4500g)
5,Adelie Penguin (Pygoscelis adeliae),3500.0,Light (3500g)
6,Gentoo penguin (Pygoscelis papua),5300.0,Heavy (5300g)
7,Adelie Penguin (Pygoscelis adeliae),3300.0,Light (3300g)
8,Chinstrap penguin (Pygoscelis antarctica),4150.0,Light (4150g)
9,Gentoo penguin (Pygoscelis papua),5650.0,Heavy (5650g)


## 2. Groupby Aggregation (Aggregate Transpilation)

The transpiler also supports custom aggregation functions in `groupby().agg()`. 

Typically, custom aggregations in pandas require downloading all groupings and computing them in Python. In BigQuery DataFrames, the transpiler compiles the Python aggregation logic directly into a BigQuery SQL aggregation expression.

Let's define a custom aggregation `range_to_count_ratio` that computes:
`ratio = (max_val - min_val) / count_val`
And apply it on the weights of the penguins grouped by `species` and `island`.


In [4]:
# Define custom aggregation logic
def range_to_count_ratio(s):
    # This UDF uses Series methods max(), min(), and count()
    return (s.max() - s.min()) / s.count()

# Run the group by and apply the custom aggregation UDF
df_agg = df.groupby(["species", "island"])["body_mass_g"].agg(range_to_count_ratio).reset_index()

# Display the aggregated result
df_agg


,species,island,body_mass_g
0,Adelie Penguin (Pygoscelis adeliae),Biscoe,43.75
1,Adelie Penguin (Pygoscelis adeliae),Dream,31.25
2,Adelie Penguin (Pygoscelis adeliae),Torgersen,35.294118
3,Chinstrap penguin (Pygoscelis antarctica),Dream,30.882353
4,Gentoo penguin (Pygoscelis papua),Biscoe,19.105691


## 3. Groupby Transform (Window Function Transpilation)

Similarly, you can use custom functions inside `groupby().transform()`. This compiles the Python logic into BigQuery SQL analytic (window) functions, broadcasting the result back to the original rows.

Let's define a custom normalization function `min_max_scale` that scales a value relative to its group's minimum and maximum values:
`scaled = (x - min) / (max - min)`

We'll use this to scale the body mass of each penguin relative only to other penguins of the *same species*.


In [5]:
# Define custom transform logic for min-max scaling
def min_max_scale(s):
    # This uses max() and min() within the group and scales each row's value
    return (s - s.min()) / (s.max() - s.min())

# Group by species and apply the custom transform UDF to body_mass_g
df_scaled = df.copy()
df_scaled["scaled_mass"] = df_scaled.groupby("species")["body_mass_g"].transform(min_max_scale)

# Display some rows to compare original body mass with the species-scaled mass
df_scaled[["species", "island", "body_mass_g", "scaled_mass"]].head(15)


,species,island,body_mass_g,scaled_mass
0,Gentoo penguin (Pygoscelis papua),Biscoe,5950.0,0.851064
1,Gentoo penguin (Pygoscelis papua),Biscoe,4400.0,0.191489
2,Adelie Penguin (Pygoscelis adeliae),Biscoe,3075.0,0.116883
3,Gentoo penguin (Pygoscelis papua),Biscoe,5000.0,0.446809
4,Gentoo penguin (Pygoscelis papua),Biscoe,4500.0,0.234043
5,Adelie Penguin (Pygoscelis adeliae),Torgersen,3500.0,0.337662
6,Gentoo penguin (Pygoscelis papua),Biscoe,5300.0,0.574468
7,Adelie Penguin (Pygoscelis adeliae),Dream,3300.0,0.233766
8,Chinstrap penguin (Pygoscelis antarctica),Dream,4150.0,0.690476
9,Gentoo penguin (Pygoscelis papua),Biscoe,5650.0,0.723404
